# 2-dars

## Import  Libraries

In [271]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [272]:
# Load Dataset
df = pd.read_csv("student-lifestyle-and-stress-dataset.csv")
df.head()

,Student_Type,Sleep_Hours,Study_Hours,Social_Media_Hours,Attendance,Exam_Pressure,Family_Support,Month,Stress_Level
0,school,6.868702,1.711722,3.176942,NaN,8.0,7.0,2.0,1
1,school,8.519088,3.251084,3.880787,93.978465,6.0,4.0,3.0,1
2,college,4.498770,6.306885,2.936172,64.421253,7.0,1.0,12.0,1
3,school,8.591223,2.384922,5.222832,81.868960,2.0,7.0,7.0,0
4,college,5.329293,9.345179,7.815869,85.847982,5.0,6.0,10.0,1


In [273]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25500 entries, 0 to 25499
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Student_Type        24248 non-null  str    
 1   Sleep_Hours         24167 non-null  float64
 2   Study_Hours         24223 non-null  float64
 3   Social_Media_Hours  24188 non-null  float64
 4   Attendance          24195 non-null  float64
 5   Exam_Pressure       24230 non-null  float64
 6   Family_Support      24209 non-null  float64
 7   Month               24186 non-null  float64
 8   Stress_Level        25500 non-null  int64  
dtypes: float64(7), int64(1), str(1)
memory usage: 1.8 MB


## Preprocessing

In [274]:
df.isnull().sum()

Student_Type          1252
Sleep_Hours           1333
Study_Hours           1277
Social_Media_Hours    1312
Attendance            1305
Exam_Pressure         1270
Family_Support        1291
Month                 1314
Stress_Level             0
dtype: int64

In [275]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from pandas.api.types import is_numeric_dtype

In [276]:
class DataPreprocessing:
    """
    Ma'lumotlarni tozalash va kodlash uchun to'liq pipeline
    """

    def __init__(self, df):
        self.df=df.copy()

    def tozala(self):
        for col in self.df.columns:
            if self.df[col].isnull().any():
                if is_numeric_dtype(self.df[col]):
                    self.df[col] = self.df[col].fillna(self.df[col].mean())
                else:
                    self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
        return self

    def encodla(self):
        encoder=LabelEncoder()
        for col in self.df.columns:
            if self.df[col].dtype=="str":
                if self.df[col].nunique()<=5:
                    dummies=pd.get_dummies(self.df[col],prefix=col,dtype=int)
                    self.df=pd.concat([self.df.drop(columns=[col]),dummies],axis=1)
                else:
                    self.df[col]=encoder.fit_transform(self.df[col])
        return self

    def scale_qil(self):
        scaler=MinMaxScaler()
        for col in self.df.columns:
            num_col = self.df.select_dtypes(include=["int64","float64"]).columns.drop("Stress_Level")
            self.df[num_col]=scaler.fit_transform(self.df[num_col])
        return self

In [277]:
dp = DataPreprocessing(df)
dp.tozala().encodla()
df = dp.df
print(f"\nPreprocessing jarayoni tugatildi")


Preprocessing jarayoni tugatildi


## Model Training

In [278]:
# Train_Test_Split
from sklearn.model_selection import train_test_split

X = df.drop("Stress_Level",axis=1)
y = df["Stress_Level"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Baseline (predict)

In [279]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(random_state=42,eval_metric="logloss",n_jobs=-1)
}

In [280]:
# Pipe line orqali baseline model
from sklearn.pipeline import Pipeline

results = []

for name, model in models.items():

    # Naive Bayes uchun StandardScaler shart emas
    if name == "Naive Bayes":
        pipeline = Pipeline([
            ("classifier", model)
        ])
    else:
        pipeline = Pipeline([
            ("scaler", MinMaxScaler()),
            ("classifier", model),
        ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average="weighted"),
        recall_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, y_pred, average="weighted")
    ])

results_df_baseline = pd.DataFrame(results, columns=[
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score"
])

In [281]:
# Tabulate

from tabulate import tabulate

print(tabulate(
    results_df_baseline,
    headers="keys",
    tablefmt="github",
    showindex=False,
    floatfmt=".3f"
))

| Model               |   Accuracy |   Precision |   Recall |   F1-score |
|---------------------|------------|-------------|----------|------------|
| Logistic Regression |      0.815 |       0.809 |    0.815 |      0.808 |
| Decision Tree       |      0.748 |       0.748 |    0.748 |      0.748 |
| Random Forest       |      0.813 |       0.807 |    0.813 |      0.806 |
| SVM                 |      0.813 |       0.808 |    0.813 |      0.802 |
| XGBoost             |      0.805 |       0.799 |    0.805 |      0.800 |


# Model Improvement

In [282]:
# Load Dataset
df = pd.read_csv("student-lifestyle-and-stress-dataset.csv")
df.head()

,Student_Type,Sleep_Hours,Study_Hours,Social_Media_Hours,Attendance,Exam_Pressure,Family_Support,Month,Stress_Level
0,school,6.868702,1.711722,3.176942,NaN,8.0,7.0,2.0,1
1,school,8.519088,3.251084,3.880787,93.978465,6.0,4.0,3.0,1
2,college,4.498770,6.306885,2.936172,64.421253,7.0,1.0,12.0,1
3,school,8.591223,2.384922,5.222832,81.868960,2.0,7.0,7.0,0
4,college,5.329293,9.345179,7.815869,85.847982,5.0,6.0,10.0,1


In [283]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25500 entries, 0 to 25499
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Student_Type        24248 non-null  str    
 1   Sleep_Hours         24167 non-null  float64
 2   Study_Hours         24223 non-null  float64
 3   Social_Media_Hours  24188 non-null  float64
 4   Attendance          24195 non-null  float64
 5   Exam_Pressure       24230 non-null  float64
 6   Family_Support      24209 non-null  float64
 7   Month               24186 non-null  float64
 8   Stress_Level        25500 non-null  int64  
dtypes: float64(7), int64(1), str(1)
memory usage: 1.8 MB


## DataPrerocessing

In [284]:
df.isnull().sum()

Student_Type          1252
Sleep_Hours           1333
Study_Hours           1277
Social_Media_Hours    1312
Attendance            1305
Exam_Pressure         1270
Family_Support        1291
Month                 1314
Stress_Level             0
dtype: int64

In [285]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from pandas.api.types import is_numeric_dtype

In [286]:
class DataPreprocessing:
    """
    Ma'lumotlarni tozalash va kodlash uchun to'liq pipeline
    """

    def __init__(self, df):
        self.df=df.copy()
        
    def encodla(self):
        encoder=LabelEncoder()
        for col in self.df.columns:
            if self.df[col].dtype=="str":
                if self.df[col].nunique()<=5:
                    dummies=pd.get_dummies(self.df[col],prefix=col,dtype=int)
                    self.df=pd.concat([self.df.drop(columns=[col]),dummies],axis=1)
                else:
                    self.df[col]=encoder.fit_transform(self.df[col])
        return self

    def scale_qil(self):
        scaler=MinMaxScaler()
        for col in self.df.columns:
            num_col = self.df.select_dtypes(include=["int64","float64"]).columns.drop("Stress_Level")
        return self

In [287]:
# Encoding (One-hot)
dp = DataPreprocessing(df)
dp.encodla()
df = dp.df

# Missing Value

### KNN Imputer

In [288]:
X = df.drop("Stress_Level", axis=1)
y = df["Stress_Level"]

# KNN Imputer
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5)
X = pd.DataFrame(imputer.fit_transform(X),columns=X.columns)
df = pd.concat([X, y], axis=1)

In [289]:
# Scaling
# dp = DataPreprocessing(df)
# dp.scale_qil()
# df = dp.df

In [290]:
# Model Training
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [291]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(random_state=42,eval_metric="logloss",n_jobs=-1)
}

In [292]:
# Pipe line orqali model
from sklearn.pipeline import Pipeline

results = []

for name, model in models.items():

    # Naive Bayes uchun StandardScaler shart emas
    if name == "Naive Bayes":
        pipeline = Pipeline([
            ("classifier", model)
        ])
    else:
        pipeline = Pipeline([
            ("scaler", MinMaxScaler()),
            ("classifier", model),
        ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average="weighted"),
        recall_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, y_pred, average="weighted")
    ])

results_df_knn = pd.DataFrame(results, columns=[
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score"
])

In [293]:
# Tabulate

from tabulate import tabulate

print(tabulate(
    results_df_knn,
    headers="keys",
    tablefmt="github",
    showindex=False,
    floatfmt=".3f"
))

| Model               |   Accuracy |   Precision |   Recall |   F1-score |
|---------------------|------------|-------------|----------|------------|
| Logistic Regression |      0.813 |       0.807 |    0.813 |      0.807 |
| Decision Tree       |      0.745 |       0.744 |    0.745 |      0.744 |
| Random Forest       |      0.815 |       0.809 |    0.815 |      0.807 |
| SVM                 |      0.810 |       0.806 |    0.810 |      0.799 |
| XGBoost             |      0.805 |       0.798 |    0.805 |      0.799 |


### MICE (Amaliy)

In [294]:
# Load Dataset
df = pd.read_csv("student-lifestyle-and-stress-dataset.csv")
df.head()

,Student_Type,Sleep_Hours,Study_Hours,Social_Media_Hours,Attendance,Exam_Pressure,Family_Support,Month,Stress_Level
0,school,6.868702,1.711722,3.176942,NaN,8.0,7.0,2.0,1
1,school,8.519088,3.251084,3.880787,93.978465,6.0,4.0,3.0,1
2,college,4.498770,6.306885,2.936172,64.421253,7.0,1.0,12.0,1
3,school,8.591223,2.384922,5.222832,81.868960,2.0,7.0,7.0,0
4,college,5.329293,9.345179,7.815869,85.847982,5.0,6.0,10.0,1


In [295]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25500 entries, 0 to 25499
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Student_Type        24248 non-null  str    
 1   Sleep_Hours         24167 non-null  float64
 2   Study_Hours         24223 non-null  float64
 3   Social_Media_Hours  24188 non-null  float64
 4   Attendance          24195 non-null  float64
 5   Exam_Pressure       24230 non-null  float64
 6   Family_Support      24209 non-null  float64
 7   Month               24186 non-null  float64
 8   Stress_Level        25500 non-null  int64  
dtypes: float64(7), int64(1), str(1)
memory usage: 1.8 MB


In [296]:
# Numeric va categorical ustunlarni ajratish
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['str']).columns

In [297]:
# MICE (Multiple Imputation by Chained Equations)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

imputer = IterativeImputer(estimator=RandomForestRegressor(n_estimators=20,
    max_depth=5,
    random_state=42,
    n_jobs=-1), max_iter=5,random_state=42)
df[num_cols] = imputer.fit_transform(df[num_cols])

# Categorical ustunlarni mode bilan to'ldirish
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Yakuniy dataframe
df_imputed = df.copy()
df_imputed = pd.DataFrame(df_imputed, columns=df.columns)

C:\Users\WIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [298]:
# Encoding ca Scaling jarayoni
dp = DataPreprocessing(df)
dp.encodla().scale_qil()
df = dp.df

In [299]:
# Model Training
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [300]:
model = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42,n_jobs=-1),
    "SVM" : SVC(),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=-1)
}

In [301]:
# Pipe line orqali model
from sklearn.pipeline import Pipeline

results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average="weighted"),
        recall_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, y_pred, average="weighted")
    ])

results_df_mice = pd.DataFrame(results, columns=[
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score"
])

In [302]:
# Tabulate

from tabulate import tabulate

print(tabulate(
    results_df_mice,
    headers="keys",
    tablefmt="github",
    showindex=False,
    floatfmt=".3f"
))

| Model               |   Accuracy |   Precision |   Recall |   F1-score |
|---------------------|------------|-------------|----------|------------|
| Logistic Regression |      0.814 |       0.809 |    0.814 |      0.808 |
| Decision Tree       |      0.745 |       0.744 |    0.745 |      0.744 |
| Random Forest       |      0.815 |       0.809 |    0.815 |      0.807 |
| SVM                 |      0.813 |       0.807 |    0.813 |      0.804 |
| XGBoost             |      0.805 |       0.798 |    0.805 |      0.799 |


## ML methods: Predictive (Model-Based) Imputation

In [303]:
# Load Dataset
df = pd.read_csv("student-lifestyle-and-stress-dataset.csv")
df.head()

,Student_Type,Sleep_Hours,Study_Hours,Social_Media_Hours,Attendance,Exam_Pressure,Family_Support,Month,Stress_Level
0,school,6.868702,1.711722,3.176942,NaN,8.0,7.0,2.0,1
1,school,8.519088,3.251084,3.880787,93.978465,6.0,4.0,3.0,1
2,college,4.498770,6.306885,2.936172,64.421253,7.0,1.0,12.0,1
3,school,8.591223,2.384922,5.222832,81.868960,2.0,7.0,7.0,0
4,college,5.329293,9.345179,7.815869,85.847982,5.0,6.0,10.0,1


In [304]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25500 entries, 0 to 25499
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Student_Type        24248 non-null  str    
 1   Sleep_Hours         24167 non-null  float64
 2   Study_Hours         24223 non-null  float64
 3   Social_Media_Hours  24188 non-null  float64
 4   Attendance          24195 non-null  float64
 5   Exam_Pressure       24230 non-null  float64
 6   Family_Support      24209 non-null  float64
 7   Month               24186 non-null  float64
 8   Stress_Level        25500 non-null  int64  
dtypes: float64(7), int64(1), str(1)
memory usage: 1.8 MB


In [305]:
# Encoding
dp = DataPreprocessing(df)
dp.encodla()
df = dp.df

### ML methods

In [306]:
from sklearn.ensemble import RandomForestRegressor
for col in df.columns:
    if df[col].isnull().sum() == 0:
        continue

    train = df[df[col].notnull()]
    test = df[df[col].isnull()]
    
    X_train = train.drop(col, axis=1)
    y_train = train[col]
    X_test = test.drop(col, axis=1)
    
    model = RandomForestRegressor()
    model.fit(X_train, y_train)
    
    # predict missing value
    df.loc[df[col].isnull(), col] = model.predict(X_test)

In [307]:
# Model Training
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [308]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(random_state=42,eval_metric="logloss",n_jobs=-1)
}

In [309]:
# Pipe line orqali model
from sklearn.pipeline import Pipeline

results = []

for name, model in models.items():

    # Naive Bayes uchun StandardScaler shart emas
    if name == "Naive Bayes":
        pipeline = Pipeline([
            ("classifier", model)
        ])
    else:
        pipeline = Pipeline([
            ("scaler", MinMaxScaler()),
            ("classifier", model),
        ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average="weighted"),
        recall_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, y_pred, average="weighted")
    ])

results_df_ml = pd.DataFrame(results, columns=[
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score"
])

In [310]:
# Tabulate

from tabulate import tabulate

print(tabulate(
    results_df_ml,
    headers="keys",
    tablefmt="github",
    showindex=False,
    floatfmt=".3f"
))

| Model               |   Accuracy |   Precision |   Recall |   F1-score |
|---------------------|------------|-------------|----------|------------|
| Logistic Regression |      0.813 |       0.807 |    0.813 |      0.807 |
| Decision Tree       |      0.745 |       0.744 |    0.745 |      0.744 |
| Random Forest       |      0.815 |       0.809 |    0.815 |      0.807 |
| SVM                 |      0.810 |       0.806 |    0.810 |      0.799 |
| XGBoost             |      0.805 |       0.798 |    0.805 |      0.799 |


## Rule-Based / Domain Knowledge Methods: Domain-Aware Imputetion

In [311]:
# Load Dataset
df = pd.read_csv("student-lifestyle-and-stress-dataset.csv")
df.head()

,Student_Type,Sleep_Hours,Study_Hours,Social_Media_Hours,Attendance,Exam_Pressure,Family_Support,Month,Stress_Level
0,school,6.868702,1.711722,3.176942,NaN,8.0,7.0,2.0,1
1,school,8.519088,3.251084,3.880787,93.978465,6.0,4.0,3.0,1
2,college,4.498770,6.306885,2.936172,64.421253,7.0,1.0,12.0,1
3,school,8.591223,2.384922,5.222832,81.868960,2.0,7.0,7.0,0
4,college,5.329293,9.345179,7.815869,85.847982,5.0,6.0,10.0,1


In [312]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25500 entries, 0 to 25499
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Student_Type        24248 non-null  str    
 1   Sleep_Hours         24167 non-null  float64
 2   Study_Hours         24223 non-null  float64
 3   Social_Media_Hours  24188 non-null  float64
 4   Attendance          24195 non-null  float64
 5   Exam_Pressure       24230 non-null  float64
 6   Family_Support      24209 non-null  float64
 7   Month               24186 non-null  float64
 8   Stress_Level        25500 non-null  int64  
dtypes: float64(7), int64(1), str(1)
memory usage: 1.8 MB


In [313]:
# DataPreprocessing
df.isnull().sum()

Student_Type          1252
Sleep_Hours           1333
Study_Hours           1277
Social_Media_Hours    1312
Attendance            1305
Exam_Pressure         1270
Family_Support        1291
Month                 1314
Stress_Level             0
dtype: int64

In [314]:
# encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["Student_Type"] = le.fit_transform(df["Student_Type"].astype(str))

In [315]:
print(type(df))
print(df.columns.tolist())

<class 'pandas.DataFrame'>
['Student_Type', 'Sleep_Hours', 'Study_Hours', 'Social_Media_Hours', 'Attendance', 'Exam_Pressure', 'Family_Support', 'Month', 'Stress_Level']


In [316]:
# Missing value RULE-BASED(Domain Knowledge)

# Rule-1
# Student_Type bo'sh bo'lsa eng ko'p uchraydigan qiymat bilan to'ldirish
df.loc[df["Student_Type"].isnull(), "Student_Type"] = df["Student_Type"].mode()[0]

# Rule-2
# Attendance bo'sh bo'lsa umumiy median bilan to'ldirish
df.loc[df["Attendance"].isnull(), "Attendance"] = df["Attendance"].median()

# Rule-3
# Sleep_Hours bo'sh bo'lsa Student_Type bo'yicha median bilan to'ldirish
mask = df["Sleep_Hours"].isnull()
df.loc[mask, "Sleep_Hours"] = (
    df.groupby("Student_Type")["Sleep_Hours"]
      .transform("median")[mask]
)

# Rule-4
# Study_Hours bo'sh bo'lsa Student_Type bo'yicha median bilan to'ldirish
mask = df["Study_Hours"].isnull()
df.loc[mask, "Study_Hours"] = (
    df.groupby("Student_Type")["Study_Hours"]
      .transform("median")[mask]
)

# Rule-5
# Social_Media_Hours bo'sh bo'lsa Student_Type bo'yicha median bilan to'ldirish
mask = df["Social_Media_Hours"].isnull()
df.loc[mask, "Social_Media_Hours"] = (
    df.groupby("Student_Type")["Social_Media_Hours"]
      .transform("median")[mask]
)

# Rule-6
# Exam_Pressure bo'sh bo'lsa Student_Type bo'yicha median bilan to'ldirish
mask = df["Exam_Pressure"].isnull()
df.loc[mask, "Exam_Pressure"] = (
    df.groupby("Student_Type")["Exam_Pressure"]
      .transform("median")[mask]
)

# Rule-7
# Family_Support bo'sh bo'lsa Student_Type bo'yicha median bilan to'ldirish
mask = df["Family_Support"].isnull()
df.loc[mask, "Family_Support"] = (
    df.groupby("Student_Type")["Family_Support"]
      .transform("median")[mask]
)

# Rule-8
# Month bo'sh bo'lsa mode bilan to'ldirish
df.loc[df["Month"].isnull(), "Month"] = df["Month"].mode()[0]

In [317]:
# Model Training
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [318]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(random_state=42,eval_metric="logloss",n_jobs=-1)
}

In [319]:
# Pipe line orqali model
from sklearn.pipeline import Pipeline

results = []

for name, model in models.items():

    # Naive Bayes uchun StandardScaler shart emas
    if name == "Naive Bayes":
        pipeline = Pipeline([
            ("classifier", model)
        ])
    else:
        pipeline = Pipeline([
            ("scaler", MinMaxScaler()),
            ("classifier", model),
        ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average="weighted"),
        recall_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, y_pred, average="weighted")
    ])

results_df_rule = pd.DataFrame(results, columns=[
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score"
])

In [320]:
# Tabulate

from tabulate import tabulate

print(tabulate(
    results_df_rule,
    headers="keys",
    tablefmt="github",
    showindex=False,
    floatfmt=".3f"
))

| Model               |   Accuracy |   Precision |   Recall |   F1-score |
|---------------------|------------|-------------|----------|------------|
| Logistic Regression |      0.813 |       0.807 |    0.813 |      0.807 |
| Decision Tree       |      0.745 |       0.744 |    0.745 |      0.744 |
| Random Forest       |      0.815 |       0.809 |    0.815 |      0.807 |
| SVM                 |      0.810 |       0.806 |    0.810 |      0.799 |
| XGBoost             |      0.805 |       0.798 |    0.805 |      0.799 |


## Implementation Strategy

In [321]:
# Load Dataset
df = pd.read_csv("student-lifestyle-and-stress-dataset.csv")
df.head()

,Student_Type,Sleep_Hours,Study_Hours,Social_Media_Hours,Attendance,Exam_Pressure,Family_Support,Month,Stress_Level
0,school,6.868702,1.711722,3.176942,NaN,8.0,7.0,2.0,1
1,school,8.519088,3.251084,3.880787,93.978465,6.0,4.0,3.0,1
2,college,4.498770,6.306885,2.936172,64.421253,7.0,1.0,12.0,1
3,school,8.591223,2.384922,5.222832,81.868960,2.0,7.0,7.0,0
4,college,5.329293,9.345179,7.815869,85.847982,5.0,6.0,10.0,1


In [322]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25500 entries, 0 to 25499
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Student_Type        24248 non-null  str    
 1   Sleep_Hours         24167 non-null  float64
 2   Study_Hours         24223 non-null  float64
 3   Social_Media_Hours  24188 non-null  float64
 4   Attendance          24195 non-null  float64
 5   Exam_Pressure       24230 non-null  float64
 6   Family_Support      24209 non-null  float64
 7   Month               24186 non-null  float64
 8   Stress_Level        25500 non-null  int64  
dtypes: float64(7), int64(1), str(1)
memory usage: 1.8 MB


In [323]:
# Categorical ustunlarni encode qilish
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Student_Type'] = le.fit_transform(df['Student_Type'])

In [324]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# KNN Imputation
imputer = KNNImputer(n_neighbors=5)

X_imputed = pd.DataFrame(
    imputer.fit_transform(X),columns=X.columns)

# Missing values tekshirish
print(X_imputed.isnull().sum())

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed,y,test_size=0.2,random_state=42)

# Pipeline
pipeline = Pipeline([
    ("model", RandomForestClassifier(random_state=42))])

# Cross Validation
scores = cross_val_score(
    pipeline,X_imputed,y,cv=5,scoring="accuracy")

print("Cross Validation Scores:", scores)
print("Average Accuracy:", scores.mean())

Sleep_Hours                     0
Study_Hours                     0
Social_Media_Hours              0
Attendance                      0
Exam_Pressure                   0
Family_Support                  0
Month                           0
Student_Type_college            0
Student_Type_school             0
Student_Type_working_student    0
dtype: int64
Cross Validation Scores: [0.82058824 0.80705882 0.80568627 0.81529412 0.81882353]
Average Accuracy: 0.8134901960784313


In [325]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(random_state=42,eval_metric="logloss",n_jobs=-1)
}

In [326]:
# Pipe line orqali model
from sklearn.pipeline import Pipeline

results = []

for name, model in models.items():

    # Naive Bayes uchun StandardScaler shart emas
    if name == "Naive Bayes":
        pipeline = Pipeline([
            ("classifier", model)
        ])
    else:
        pipeline = Pipeline([
            ("scaler", MinMaxScaler()),
            ("classifier", model),
        ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average="weighted"),
        recall_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, y_pred, average="weighted")
    ])

results_df_IS = pd.DataFrame(results, columns=[
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score"
])

In [327]:
# Tabulate

from tabulate import tabulate

print(tabulate(
    results_df_IS,
    headers="keys",
    tablefmt="github",
    showindex=False,
    floatfmt=".3f"
))

| Model               |   Accuracy |   Precision |   Recall |   F1-score |
|---------------------|------------|-------------|----------|------------|
| Logistic Regression |      0.813 |       0.807 |    0.813 |      0.807 |
| Decision Tree       |      0.745 |       0.744 |    0.745 |      0.744 |
| Random Forest       |      0.815 |       0.809 |    0.815 |      0.807 |
| SVM                 |      0.810 |       0.806 |    0.810 |      0.799 |
| XGBoost             |      0.805 |       0.798 |    0.805 |      0.799 |


In [328]:
# results_df_baseline, results_df_knn, results_df_mice,results_df_ml, results_df_rule, results_df_IS

In [330]:
import pandas as pd
from tabulate import tabulate

# Har bir jadvalga Missing Value Method ustunini qo'shamiz
results_df_baseline["Missing Value Method"] = "Baseline"
results_df_knn["Missing Value Method"] = "KNN"
results_df_mice["Missing Value Method"] = "MICE"
results_df_ml["Missing Value Method"] = "Predictive"
results_df_rule["Missing Value Method"] = "Rule-Based"
results_df_IS["Missing Value Method"] = "Hybrid"

# Hammasini birlashtirish
final_results = pd.concat([
    results_df_baseline,
    results_df_knn,
    results_df_mice,
    results_df_ml,
    results_df_rule,
    results_df_IS
], ignore_index=True)

# Ustunlarni tartiblash
final_results = final_results[
    [
        "Missing Value Method",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ]
]

# Eng yaxshi natijalar tepada chiqishi uchun
final_results = final_results.sort_values(
    by="F1-score",
    ascending=False
)

print(tabulate(
    final_results,
    headers="keys",
    tablefmt="github",
    showindex=False
))

| Missing Value Method   | Model               |   Accuracy |   Precision |   Recall |   F1-score |
|------------------------|---------------------|------------|-------------|----------|------------|
| MICE                   | Logistic Regression |   0.814314 |    0.808697 | 0.814314 |   0.808244 |
| Baseline               | Logistic Regression |   0.81451  |    0.808886 | 0.81451  |   0.808127 |
| MICE                   | Random Forest       |   0.81451  |    0.808918 | 0.81451  |   0.80737  |
| KNN                    | Random Forest       |   0.814706 |    0.809167 | 0.814706 |   0.807342 |
| Rule-Based             | Random Forest       |   0.814706 |    0.809167 | 0.814706 |   0.807342 |
| Predictive             | Random Forest       |   0.814706 |    0.809167 | 0.814706 |   0.807342 |
| Hybrid                 | Random Forest       |   0.814706 |    0.809167 | 0.814706 |   0.807342 |
| Predictive             | Logistic Regression |   0.812941 |    0.807197 | 0.812941 |   0.806654 |


In [331]:
import pandas as pd

# Har bir jadvalga Missing Value Method ustunini qo'shamiz
results_df_baseline["Missing Value Method"] = "Baseline"
results_df_knn["Missing Value Method"] = "KNN"
results_df_mice["Missing Value Method"] = "MICE"
results_df_ml["Missing Value Method"] = "Predictive"
results_df_rule["Missing Value Method"] = "Rule-Based"
results_df_IS["Missing Value Method"] = "Implementation Strategy"

# Birlashtirish
final_results = pd.concat([
    results_df_baseline,
    results_df_knn,
    results_df_mice,
    results_df_ml,
    results_df_rule,
    results_df_IS
], ignore_index=True)

# Ustunlarni tartiblash
final_results = final_results[
    ["Missing Value Method",
     "Model",
     "Accuracy",
     "Precision",
     "Recall",
     "F1-score"]
]

# F1-score bo'yicha saralash
final_results = final_results.sort_values(
    by="F1-score",
    ascending=False
).reset_index(drop=True)

# Jadvalni chiroyli ko'rsatish
display(
    final_results.style
    .background_gradient(
        subset=["Accuracy","Precision","Recall","F1-score"],
        cmap="YlGn"
    )
    .format({
        "Accuracy":"{:.4f}",
        "Precision":"{:.4f}",
        "Recall":"{:.4f}",
        "F1-score":"{:.4f}"
    })
)

,Missing Value Method,Model,Accuracy,Precision,Recall,F1-score
0,MICE,Logistic Regression,0.8143,0.8087,0.8143,0.8082
1,Baseline,Logistic Regression,0.8145,0.8089,0.8145,0.8081
2,MICE,Random Forest,0.8145,0.8089,0.8145,0.8074
3,KNN,Random Forest,0.8147,0.8092,0.8147,0.8073
4,Rule-Based,Random Forest,0.8147,0.8092,0.8147,0.8073
5,Predictive,Random Forest,0.8147,0.8092,0.8147,0.8073
6,Implementation Strategy,Random Forest,0.8147,0.8092,0.8147,0.8073
7,Predictive,Logistic Regression,0.8129,0.8072,0.8129,0.8067
8,Implementation Strategy,Logistic Regression,0.8129,0.8072,0.8129,0.8067
9,Rule-Based,Logistic Regression,0.8129,0.8072,0.8129,0.8067
